# NB1: Baseline

Relevant to chapter 9.3.3 from the Understanding Deep Learning Book.

**Theory Claim**: Dropout removes kinks in regions far from the training data that don't affect the loss. With more training data, the training points cover the input space more desnely, so there are fewere unconstrained regions where those kings can form.

**Our question**: Does the improvement in performance of neural networks hold when using dropout regularisation with increasing training dataset sizes?

## Experiment Information

**Our Baseline**: Overffited dataset with increasingly applied levels of dropout.
To force overfitting we have:
-  Used ... percent of training data.
- Used FashionMNIST as it has hard to seperate classes (explained below).
 - Every model has 256 hidden units and 2 hidden layers so there are more weights than data (roughly 45 per image).
 - Optimiser Adam is used so that the model fits quickly and with 20 epochs, memorising will have occured.
Therefore, we have guaranteed overfitting which is vital to the baseline as we want to show that dropout improves models performance.


Fashion-MNIST (70,000 grayscale images across 10 clothing categories).
- Fixed image sizes of 28x28 pixels.
- Up to 60,000 training images allows for a range in test cases when testing the breaking point of the claim
- Clothing categories are visually similar making classification more difficult


Run with over 5 randomly selected seeds. Mean and standard deviation calculated for a final representation of the result.

## Initial Experiment Set Up

In [90]:
%pip install torch
%pip install torchvision

#libray imports
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torchvision import datasets, transforms

device = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')

transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)
print(f'FashionMNIST: {len(train_dataset)} train images, {len(test_dataset)} test images')

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
FashionMNIST: 60000 train images, 10000 test images


### Nuisance Parameters

Controlled settings that are fixed for the experiment. 

Model Explanation:
256 hidden units gives:
- Input Layer: 784 neurons
- Hidden Layer 1: 256 neurons
- Hidden Layer 2: 256 neurons
- Output Layer: 10 neurons

This layout is complex enough to learn patterns in Fashion-MNIST.

In [91]:
#Settings
HIDDEN_UNITS = 256
TRAIN_PERCENT = 0.1
N_TRAIN = int(len(train_dataset)* TRAIN_PERCENT)
EPOCHS = 20 
BATCH = 128 #Mini-batch (balance between computation efficient and gradient stability)
LEARNING_RATE = 0.001 #Adam's standard
DROPOUT_RATES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
SEED = [16, 31, 9, 92, 2]

## Tensor Loading

In [ ]:
#Loading the dataset into device tensors: ... of the dataset will be used for training

def to_tensor(dataset, index=None):
    images, labels = (dataset.data, dataset.targets) if index is None else (dataset.data[index], dataset.targets[index])
    X = images.float().div(255).view(-1,784)
    return X.to(device), labels.to(device)

#generator = torch.Generator().manual_seed(16)
def random_sample(train_dataset, test_dataset):
    order = torch.randperm(len(train_dataset))[:N_TRAIN]
    x_train, y_train = to_tensor(train_dataset, order)
    x_test, y_test = to_tensor(test_dataset)
    print(f"device {device} | train {tuple(x_train.shape)} | test {tuple(x_test.shape)}")
    return x_train, x_test, y_train, y_test

## Training

In [93]:
#Making and training the model

#Why did we choose 2 hidden layers? we could change this ..., standardise this ... or explain why 2 is the best option.
def make_model(dropout_rate, seed):
    torch.manual_seed(seed)
    return torch.nn.Sequential(
        torch.nn.Linear(784, HIDDEN_UNITS), torch.nn.ReLU(), torch.nn.Dropout(dropout_rate),
        torch.nn.Linear(HIDDEN_UNITS, HIDDEN_UNITS), torch.nn.ReLU(), torch.nn.Dropout(dropout_rate),
        torch.nn.Linear(HIDDEN_UNITS, 10) #There are only 10 types of clothing class in fashionMNIST.
    ).to(device)

@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    return(model(X).argmax(1) == y).float().mean().item()

@torch.no_grad()
def calculate_loss(model, X, y):
    model.eval()
    loss_function = nn.CrossEntropyLoss()
    return loss_function(model(X), y).item()

def train(model, x_train, y_train, x_test, y_test, seed, verbose=True):
    random_generator = torch.Generator(device="cpu").manual_seed(seed)
    adam_opt = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE)
    loss_function = nn.CrossEntropyLoss()
    results = {"train": [], 
               "test": [],
               "train_loss": [],
               "test_loss": []}
    for ep in range(EPOCHS):
        model.train()
        shuffle = torch.randperm(len(x_train), generator=random_generator).to(device)
        for i in range(0, len(x_train), BATCH):
            index = shuffle[i:i + BATCH]
            adam_opt.zero_grad()
            loss = loss_function(model(x_train[index]), y_train[index])
            loss.backward()
            adam_opt.step()
        results["train"].append(accuracy(model, x_train, y_train))
        results["test"].append(accuracy(model, x_test, y_test))
        results["train_loss"].append(calculate_loss(model, x_train, y_train))
        results["test_loss"].append(calculate_loss(model, x_test, y_test))
        if verbose and (ep + 1) % 10  ==0:
            print(
                f"epoch {ep + 1} | "
                f"Training Accuracy: {results['train'][-1]:.4f} | "
                f"Testing Accuracy: {results['test'][-1]:.4f} | "
                f"Training loss: {results['train_loss'][-1]:.4f} | "
                f"Testing Loss: {results['test_loss'][-1]:.4f}"
                
            )
    return results

## Running the Experiment

In [94]:
#Run with varying dropout
def dropout_experiment(seed):
    models = {}
    results = {}
    x_train, x_test, y_train, y_test = random_sample(train_dataset, test_dataset)

    for r in DROPOUT_RATES:
        models[r] = make_model(r, seed)
        results[r] = train(models[r], x_train, y_train, x_test, y_test, seed)

    print(f"{'Dropout Rate'} {'Training Accuracy'} {'Test Accuracy'} {'Difference'} {'Training Loss'} {'Testing Loss'}")
    for r in DROPOUT_RATES:
        train_data = results[r]["train"][-1]
        test_data = results[r]["test"][-1]
        train_loss = results[r]["train_loss"][-1]
        test_loss = results[r]["test_loss"][-1]
        print(f"{r:12.2f} {train_data:17.4f} {test_data:13.4f} {train_data-test_data:10.4f} {train_loss:12.4f} {test_loss:13.4f}")

    return results

In [95]:
def main():
    results = {}
    for i in range(0, len(SEED)):
        print(f"\nSeed: {SEED[i]}")
        results[SEED[i]] = dropout_experiment(SEED[i])
    return results

results = main()


Seed: 16


device mps | train (6000, 784) | test (10000, 784)
epoch 10 | Training Accuracy: 0.8998 | Testing Accuracy: 0.8399 | Training loss: 0.2962 | Testing Loss: 0.4527
epoch 20 | Training Accuracy: 0.9092 | Testing Accuracy: 0.8257 | Training loss: 0.2378 | Testing Loss: 0.5525
epoch 10 | Training Accuracy: 0.8937 | Testing Accuracy: 0.8383 | Training loss: 0.3034 | Testing Loss: 0.4442
epoch 20 | Training Accuracy: 0.9192 | Testing Accuracy: 0.8401 | Training loss: 0.2167 | Testing Loss: 0.4889
epoch 10 | Training Accuracy: 0.8862 | Testing Accuracy: 0.8358 | Training loss: 0.3158 | Testing Loss: 0.4512
epoch 20 | Training Accuracy: 0.9188 | Testing Accuracy: 0.8453 | Training loss: 0.2237 | Testing Loss: 0.4743
epoch 10 | Training Accuracy: 0.8827 | Testing Accuracy: 0.8313 | Training loss: 0.3340 | Testing Loss: 0.4623
epoch 20 | Training Accuracy: 0.9080 | Testing Accuracy: 0.8465 | Training loss: 0.2449 | Testing Loss: 0.4557
epoch 10 | Training Accuracy: 0.8702 | Testing Accuracy: 0.82

## Final Result Comparision

In [ ]:
#Code for mean and standard deviation across all results
results_final = {}
columns = ['Training Accuracy', 'Test Accuracy', 'Difference', 'Training Loss', 'Testing Loss']

def format(d):
    return {
        'Training Accuracy': d['train'][-1],
        'Test Accuracy' : d['test'][-1],
        'Difference' : d['train'][-1] - d['test'][-1],
        'Training Loss': d['train_loss'][-1],
        'Testing Loss': d['test_loss'][-1]
    }

for r in DROPOUT_RATES:
    results_final[r] = {}
    for i in range(0, 5):
        values = [format(results[s][r])[columns[i]] for s in SEED]
        results_final[r][columns[i]] = (np.mean(values), np.std(values, ddof=1))

print("Table displaying mean and standard deviation across all 5 seeds.")
print(f"{'Dropout Rate'} | {'Training Accuracy'} | {'Test Accuracy':17} | {'Difference':17} | {'Training Loss':17} | {'Testing Loss':17}")
for r in DROPOUT_RATES:
    print(f"{r:12.2f} | {results_final[r][columns[0]][0]:.4f}{results_final[r][columns[0]][1]:11.4f} | {results_final[r][columns[1]][0]:.4f}{results_final[r][columns[1]][1]:11.4f} | {results_final[r][columns[2]][0]:.4f}{results_final[r][columns[2]][1]:11.4f} | {results_final[r][columns[3]][0]:.4f}{results_final[r][columns[3]][1]:11.4f} | {results_final[r][columns[4]][0]:.4f}{results_final[r][columns[4]][1]:11.4f}")

Table displaying mean and standard deviation across all 5 seeds.
Dropout Rate | Training Accuracy | Test Accuracy     | Difference        | Training Loss     | Testing Loss     
        0.00 | 0.9228     0.0138 | 0.8367     0.0080 | 0.0862     0.0070 | 0.2106     0.0321 | 0.5080     0.0269
        0.10 | 0.9227     0.0090 | 0.8437     0.0043 | 0.0791     0.0074 | 0.2081     0.0233 | 0.4773     0.0152
        0.20 | 0.9180     0.0050 | 0.8457     0.0034 | 0.0723     0.0071 | 0.2231     0.0163 | 0.4665     0.0154
        0.30 | 0.9104     0.0100 | 0.8467     0.0058 | 0.0637     0.0096 | 0.2415     0.0235 | 0.4603     0.0215
        0.40 | 0.9014     0.0110 | 0.8438     0.0075 | 0.0576     0.0094 | 0.2656     0.0235 | 0.4613     0.0229
        0.50 | 0.8916     0.0070 | 0.8425     0.0054 | 0.0492     0.0074 | 0.2885     0.0212 | 0.4547     0.0143
        0.60 | 0.8815     0.0100 | 0.8383     0.0062 | 0.0431     0.0081 | 0.3196     0.0206 | 0.4565     0.0129
        0.70 | 0.8654     0.009

## Conclusion
